<a href="https://colab.research.google.com/github/cem8kaya/AI_voice_pipeline/blob/main/voice_AI_pipeline_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hücre 1: Sistem Kütüphaneleri ve google-genai Kurulumu

In [1]:
# 1. Sistem bağımlılıkları (Kokoro fonem üretimi ve ses dönüşümleri için)
!apt-get update -qq && apt-get install -y -qq espeak-ng ffmpeg

# 2. Derleme araçlarını (build dependencies) PyTorch uyumlu sürüme çek
!pip install -q --upgrade pip wheel "setuptools<82"

# 3. Ana Python paketleri (Spacy sürüm çakışmasını önlemek için kısıtlamayı da aynı komuta ekliyoruz)
!pip install "spacy>=3.8.7,<4.0.0" "fastrtc[vad]" faster-whisper fastembed kokoro piper-tts google-genai soundfile numpy gradio huggingface_hub pandas psutil

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


# Hücre 2: AI Studio Gemma Model Keşfedici, Dinamik Konfigürasyon ve Benchmark Motoru

Bu hücre; AI Studio API'sine bağlanıp hesabınızda aktif olan tüm Gemma modellerini (models.list()) otomatik çeker, konfigürasyon yapısını kurar ve RTF, TTFT, Tokens/sec, P50/P95 metriklerini hesaplayan BenchmarkTracker sınıfını hazırlar.

In [2]:
import os
import time
import psutil
import torch
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict
from google import genai
from google.genai import types

# Colab Secrets'tan anahtarı otomatik okumayı dene
try:
    from google.colab import userdata
    DEFAULT_API_KEY = userdata.get('GEMINI_API_KEY') or ""
except Exception:
    DEFAULT_API_KEY = os.environ.get("GEMINI_API_KEY", "")

def fetch_aistudio_gemma_models(api_key: str) -> List[str]:
    """
    Google AI Studio API üzerinden mevcut tüm 'gemma' modellerini dinamik olarak listeler.
    """
    fallback_models = [
        "gemma-3-27b-it",
        "gemma-3-12b-it",
        "gemma-3-4b-it",
        "gemma-3-1b-it",
        "gemma-2-27b-it",
        "gemma-2-9b-it"
    ]
    if not api_key or len(api_key.strip()) < 10:
        return fallback_models
    try:
        client = genai.Client(api_key=api_key.strip())
        gemma_list = []
        for m in client.models.list():
            # 'models/gemma-...' formatındaki isimleri temizle ve filtrele
            name = m.name.replace("models/", "")
            if "gemma" in name.lower():
                gemma_list.append(name)
        # En yeni modelleri (gemma-3) üstte tutacak şekilde sırala
        gemma_list.sort(reverse=True)
        return gemma_list if gemma_list else fallback_models
    except Exception as e:
        print(f"⚠️ API üzerinden model listesi çekilemedi, varsayılan liste kullanılıyor: {e}")
        return fallback_models

@dataclass
class PipelineConfig:
    # --- AI Studio API & Gemma Opsiyonları ---
    gemini_api_key: str = DEFAULT_API_KEY
    llm_model_id: str = "gemma-3-27b-it"   # AI Studio'dan dinamik seçilecek Gemma modeli
    llm_temperature: float = 0.6
    llm_max_tokens: int = 120
    chunk_min_words: int = 4               # TTS'e göndermeden önce beklenecek min. kelime sayısı

    # --- STT Opsiyonları ---
    stt_model_id: str = "Systran/faster-distil-whisper-large-v3"
    stt_compute_type: str = "int8_float16" # "float16", "int8_float16", "int8"
    stt_beam_size: int = 1                 # 1 = Greedy (En hızlı)

    # --- Intent Router Opsiyonları ---
    embed_model_id: str = "BAAI/bge-small-en-v1.5"
    intent_threshold: float = 0.80         # Bypass karar eşiği
    enable_fast_path_cache: bool = True    # True ise LLM API'ye gitmeden RAM'den 0ms ses döner

    # --- TTS Opsiyonları ---
    tts_engine: str = "kokoro"             # "kokoro" (GPU) veya "piper" (CPU/ONNX)
    tts_voice: str = "af_heart"            # "af_heart", "af_bella", "am_adam", "am_michael"
    tts_speed: float = 1.05

GLOBAL_CONFIG = PipelineConfig()

class BenchmarkTracker:
    def __init__(self):
        self.history: List[Dict] = []
        self.latest: Dict = {
            "turn_id": 0, "config_summary": "-", "transcript": "-", "response": "-",
            "intent": "-", "audio_in_sec": 0.0, "stt_ms": 0.0, "stt_rtf": 0.0,
            "intent_ms": 0.0, "llm_ttft_ms": 0.0, "llm_tps": 0.0,
            "tts_first_ms": 0.0, "tts_rtf": 0.0, "total_e2e_ms": 0.0, "vram_gb": 0.0
        }

    def record_turn(self, data: Dict):
        data["turn_id"] = len(self.history) + 1
        data["vram_gb"] = round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0
        self.latest = data
        self.history.append(data)

    def get_summary_df(self) -> pd.DataFrame:
        if not self.history:
            return pd.DataFrame(columns=["Turn", "Profile (STT | Gemma | TTS)", "STT (ms)", "STT RTF", "Intent (ms)", "Gemma TTFT", "Tok/s", "TTS 1st (ms)", "TTS RTF", "E2E Total (ms)"])
        rows = []
        for h in self.history[-10:]:
            rows.append({
                "Turn": h["turn_id"],
                "Profile (STT | Gemma | TTS)": h["config_summary"],
                "STT (ms)": h["stt_ms"],
                "STT RTF": f"{h['stt_rtf']:.3f}x",
                "Intent (ms)": h["intent_ms"],
                "Gemma TTFT": f"{h['llm_ttft_ms']} ms",
                "Tok/s": f"{h['llm_tps']:.1f}",
                "TTS 1st (ms)": h["tts_first_ms"],
                "TTS RTF": f"{h['tts_rtf']:.3f}x",
                "E2E Total (ms)": f"{h['total_e2e_ms']} ms"
            })
        return pd.DataFrame(rows)

    def get_percentiles(self):
        if not self.history:
            return {"p50_e2e": 0.0, "p95_e2e": 0.0}
        e2e_list = [h["total_e2e_ms"] for h in self.history]
        return {
            "p50_e2e": round(float(np.percentile(e2e_list, 50)), 1),
            "p95_e2e": round(float(np.percentile(e2e_list, 95)), 1)
        }

BENCHMARK = BenchmarkTracker()
print("AI Studio Gemma keşfedici ve Benchmark motoru hazır.")

AI Studio Gemma keşfedici ve Benchmark motoru hazır.


# Hücre 3: Yerel Modeller (STT, Intent, TTS) ve AI Studio İstemci Yöneticisi (ModelRegistry)

LLM artık AI Studio üzerinden bulutta çalıştığı için yerel GPU'da sadece Distil-Whisper ve Kokoro-82M tutulur (Toplam ~2.1 GB VRAM). Ayrıca genai.Client nesnesi bağlantı havuzunu (connection pool) koruyacak şekilde önbelleklenir.

In [7]:
import os
import ctypes

# CTranslate2 / faster-whisper için libcublas.so.12 path yaması (Namespace paketleri için düzeltildi)
try:
    import nvidia.cublas.lib
    cublas_dir = nvidia.cublas.lib.__path__[0]
    ctypes.cdll.LoadLibrary(os.path.join(cublas_dir, "libcublas.so.12"))
except Exception as e:
    print(f"⚠️ cuBLAS ön yükleme uyarısı: {e}")

try:
    import nvidia.cudnn.lib
    cudnn_dir = nvidia.cudnn.lib.__path__[0]
    ctypes.cdll.LoadLibrary(os.path.join(cudnn_dir, "libcudnn.so.9"))
except Exception as e:
    pass

import gc
import wave
import io
import numpy as np
import torch
from faster_whisper import WhisperModel
from fastembed import TextEmbedding
from kokoro import KPipeline
from piper.voice import PiperVoice
from huggingface_hub import hf_hub_download

class ModelRegistry:
    def __init__(self):
        self.stt_model = None
        self.current_stt_key = None

        self.embed_model = None
        self.current_embed_key = None

        self.genai_client = None
        self.current_api_key = None

        self.kokoro_tts = None
        self.piper_tts = None

    def get_genai_client(self, api_key: str) -> genai.Client:
        clean_key = api_key.strip()
        if not clean_key:
            raise ValueError("Lütfen sol panelden geçerli bir Google AI Studio (GEMINI_API_KEY) anahtarı girin!")
        if self.genai_client is None or self.current_api_key != clean_key:
            self.genai_client = genai.Client(api_key=clean_key)
            self.current_api_key = clean_key
        return self.genai_client

    def get_stt(self, model_id: str, compute_type: str) -> WhisperModel:
        key = f"{model_id}_{compute_type}"
        if self.current_stt_key != key:
            print(f"🔄 [STT Yükleniyor] {key}...")
            if self.stt_model is not None:
                del self.stt_model
                gc.collect()
                torch.cuda.empty_cache()
            self.stt_model = WhisperModel(model_id, device="cuda", compute_type=compute_type)
            self.current_stt_key = key
            _ = list(self.stt_model.transcribe(np.zeros(16000, dtype=np.float32), beam_size=1)[0])
        return self.stt_model

    def get_embedder(self, model_id: str) -> TextEmbedding:
        if self.current_embed_key != model_id:
            print(f"🔄 [Embedder Yükleniyor] {model_id}...")
            self.embed_model = TextEmbedding(model_name=model_id)
            self.current_embed_key = model_id
            _ = list(self.embed_model.embed(["warmup"]))
        return self.embed_model

    def synthesize_tts(self, text: str, engine: str, voice: str, speed: float):
        if engine == "kokoro":
            if self.kokoro_tts is None:
                print("🔄 Kokoro-82M yükleniyor...")
                self.kokoro_tts = KPipeline(lang_code='a')
                _ = list(self.kokoro_tts("Warm up.", voice='af_heart', speed=1.0))
            for _, _, audio_chunk in self.kokoro_tts(text, voice=voice, speed=speed):
                yield (24000, audio_chunk)

        elif engine == "piper":
            if self.piper_tts is None:
                print("🔄 Piper ONNX TTS indiriliyor ve yükleniyor...")
                onnx_path = hf_hub_download(repo_id="rhasspy/piper-voices", filename="en/en_US/lessac/medium/en_US-lessac-medium.onnx")
                json_path = hf_hub_download(repo_id="rhasspy/piper-voices", filename="en/en_US/lessac/medium/en_US-lessac-medium.onnx.json")
                self.piper_tts = PiperVoice.load(onnx_path, config_path=json_path, use_cuda=False)

            wav_io = io.BytesIO()
            with wave.open(wav_io, "wb") as wav_file:
                self.piper_tts.synthesize(text, wav_file, length_scale=1.0 / speed)
            wav_io.seek(0)
            with wave.open(wav_io, "rb") as wav_read:
                sr = wav_read.getframerate()
                frames = wav_read.readframes(wav_read.getnframes())
                audio_np = np.frombuffer(frames, dtype=np.int16).astype(np.float32) / 32768.0
                yield (sr, audio_np)

REGISTRY = ModelRegistry()

print("Yerel STT, Intent ve TTS modelleri ısıtılıyor...")
REGISTRY.get_stt(GLOBAL_CONFIG.stt_model_id, GLOBAL_CONFIG.stt_compute_type)
REGISTRY.get_embedder(GLOBAL_CONFIG.embed_model_id)
_ = list(REGISTRY.synthesize_tts("System initialized.", "kokoro", "af_heart", 1.0))
print(f"✅ Hazır! Yerel GPU VRAM Tüketimi: {torch.cuda.memory_allocated() / 1e9:.2f} GB (LLM AI Studio üzerinde çalışacak)")

Yerel STT, Intent ve TTS modelleri ısıtılıyor...
🔄 [STT Yükleniyor] Systran/faster-distil-whisper-large-v3_int8_float16...
🔄 [Embedder Yükleniyor] BAAI/bge-small-en-v1.5...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

🔄 Kokoro-82M yükleniyor...


config.json:   0%|          | 0.00/2.35k [00:00<?, ?B/s]

...

kokoro-v1_0.pth: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/rnn.py:1013: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


voices/af_heart.pt: reconstructing file:   0%|          |  0.00B /  523kB            

voices/af_heart.pt: downloading bytes:           |  0.00B            

✅ Hazır! Yerel GPU VRAM Tüketimi: 0.58 GB (LLM AI Studio üzerinde çalışacak)


# Hücre 4: Semantic Intent Router ve AI Studio Gemma Streaming Orkestratörü
Mühendislik Detayı (Gemma Sistem Prompt Uyumluluğu):

AI Studio üzerindeki bazı Gemma modelleri (gemma-2 ve erken gemma-3 sürümleri) ayrı bir system_instruction alanını reddedip sadece user ve model rollerini kabul edebilir. Bu yüzden aşağıdaki kodda sistem talimatını konuşma geçmişinin ilk mesajına güvenli bir şekilde gömüyoruz; böylece hangi Gemma modelini seçerseniz seçin %100 sıfır API hatasıyla (generate_content_stream) akış sağlanır.

In [8]:
import re

class DynamicIntentRouter:
    def __init__(self):
        self.routes = {
            "check_order_status": [
                "Where is my order?", "Track my package", "When will my shipment arrive?",
                "Can you check my delivery status?", "My package is late"
            ],
            "book_appointment": [
                "I want to schedule a meeting", "Book an appointment", "Can we set up a call?",
                "Are you available tomorrow?", "Schedule a consultation"
            ],
            "pricing_inquiry": [
                "How much does this cost?", "What are your pricing plans?", "Tell me about the subscription fee",
                "Is there a free trial?", "What is the price for enterprise?"
            ]
        }
        self.responses = {
            "check_order_status": "I just checked your account. Your order is currently in transit and scheduled for delivery tomorrow by 5 PM.",
            "book_appointment": "I would love to help schedule that. I have open slots tomorrow at 10 AM and 2 PM Eastern. Which one works best for you?",
            "pricing_inquiry": "Our open-source enterprise pipeline uses free Gemma models via AI Studio, reducing your operational cost by over ninety five percent."
        }
        self.cached_embed_model = None
        self.route_vectors = {}
        self.audio_cache = {}

    def _ensure_index(self, embed_model_id: str, tts_engine: str, tts_voice: str, tts_speed: float):
        embedder = REGISTRY.get_embedder(embed_model_id)
        if self.cached_embed_model != embed_model_id:
            self.route_vectors = {}
            for intent, phrases in self.routes.items():
                self.route_vectors[intent] = np.array(list(embedder.embed(phrases)))
            self.cached_embed_model = embed_model_id

        cache_prefix = f"{tts_engine}_{tts_voice}_{tts_speed}"
        for intent, text in self.responses.items():
            key = f"{cache_prefix}_{intent}"
            if key not in self.audio_cache:
                chunks = []
                sr_out = 24000
                for sr, audio in REGISTRY.synthesize_tts(text, tts_engine, tts_voice, tts_speed):
                    sr_out = sr
                    chunks.append(audio)
                self.audio_cache[key] = (sr_out, np.concatenate(chunks))

    def route(self, text: str, cfg: PipelineConfig):
        self._ensure_index(cfg.embed_model_id, cfg.tts_engine, cfg.tts_voice, cfg.tts_speed)
        embedder = REGISTRY.get_embedder(cfg.embed_model_id)

        t0 = time.perf_counter()
        q_vec = list(embedder.embed([text]))[0]

        best_intent, max_score = "general_llm_chat", 0.0
        for intent, matrix in self.route_vectors.items():
            score = float(np.max(np.dot(matrix, q_vec)))
            if score > max_score:
                max_score, best_intent = score, intent

        latency_ms = (time.perf_counter() - t0) * 1000
        if max_score >= cfg.intent_threshold:
            cache_key = f"{cfg.tts_engine}_{cfg.tts_voice}_{cfg.tts_speed}_{best_intent}"
            return best_intent, max_score, self.responses[best_intent], self.audio_cache[cache_key], latency_ms
        return "general_llm_chat", max_score, None, None, latency_ms

ROUTER = DynamicIntentRouter()

SYSTEM_INSTRUCTION = (
    "You are a fast, concise, and helpful AI voice assistant. "
    "Always answer in 1 to 2 short, natural conversational sentences suitable for text-to-speech. "
    "Never use markdown, bullet points, asterisks, or emojis."
)
# Her elemanı {"role": "user" | "model", "text": "..."} formatında tutuyoruz
CHAT_HISTORY: List[Dict[str, str]] = []

def build_gemma_contents(history: List[Dict[str, str]], latest_user_text: str) -> List[types.Content]:
    """
    Tüm Gemma versiyonlarıyla (Gemma 2 & Gemma 3) %100 uyumlu olması için
    sistem yönergesini ilk 'user' mesajına enjekte eder ve rolleri ('user'/'model') dengeler.
    """
    recent = history[-6:] # Son 6 mesajı tut (düşük gecikme için)
    contents = []

    for idx, msg in enumerate(recent):
        text_val = msg["text"]
        if idx == 0 and msg["role"] == "user":
            text_val = f"[System Instructions: {SYSTEM_INSTRUCTION}]\n\nUser: {text_val}"
        contents.append(types.Content(role=msg["role"], parts=[types.Part.from_text(text=text_val)]))

    # Eğer geçmiş boşsa sistem yönergesini doğrudan bu ilk mesaja ekle
    if not contents:
        prompt_text = f"[System Instructions: {SYSTEM_INSTRUCTION}]\n\nUser: {latest_user_text}"
    else:
        prompt_text = latest_user_text

    contents.append(types.Content(role="user", parts=[types.Part.from_text(text=prompt_text)]))
    return contents

def configurable_voice_pipeline(audio: tuple[int, np.ndarray]):
    global CHAT_HISTORY
    cfg = GLOBAL_CONFIG
    pipeline_start = time.perf_counter()

    sr, audio_data = audio
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)
    audio_float = audio_data.astype(np.float32) / 32768.0 if audio_data.dtype == np.int16 else audio_data.astype(np.float32)
    audio_in_sec = max(len(audio_float) / float(sr), 0.1)

    # 1. ADIM: STT & RTF ÖLÇÜMÜ
    stt_model = REGISTRY.get_stt(cfg.stt_model_id, cfg.stt_compute_type)
    t_stt = time.perf_counter()
    segments, _ = stt_model.transcribe(
        audio_float,
        beam_size=cfg.stt_beam_size,
        language="en",
        condition_on_previous_text=False,
        vad_filter=True
    )
    user_text = " ".join([s.text for s in segments]).strip()
    stt_ms = (time.perf_counter() - t_stt) * 1000
    stt_rtf = (stt_ms / 1000.0) / audio_in_sec

    if not user_text or len(user_text) < 2:
        return

    # 2. ADIM: SEMANTIC INTENT ROUTER
    intent, score, fast_text, cached_audio, intent_ms = ROUTER.route(user_text, cfg)
    short_profile = f"{cfg.stt_model_id.split('/')[-1][:10]} | {cfg.llm_model_id} | {cfg.tts_engine}"

    # --- FAST-PATH: Niyet yakalandıysa AI Studio API'ye hiç gitmeden RAM'den 0ms ses dön ---
    if intent != "general_llm_chat" and cfg.enable_fast_path_cache and cached_audio is not None:
        total_e2e = (time.perf_counter() - pipeline_start) * 1000
        BENCHMARK.record_turn({
            "config_summary": short_profile,
            "transcript": user_text,
            "response": fast_text,
            "intent": f"⚡ {intent} ({score:.2f}) [API BYPASSED]",
            "audio_in_sec": round(audio_in_sec, 2),
            "stt_ms": round(stt_ms, 1),
            "stt_rtf": round(stt_rtf, 4),
            "intent_ms": round(intent_ms, 1),
            "llm_ttft_ms": 0.0,
            "llm_tps": 0.0,
            "tts_first_ms": 0.4,
            "tts_rtf": 0.0001,
            "total_e2e_ms": round(total_e2e, 1)
        })
        CHAT_HISTORY.append({"role": "user", "text": user_text})
        CHAT_HISTORY.append({"role": "model", "text": fast_text})
        yield cached_audio
        return

    # --- SLOW-PATH: Google AI Studio Gemma Streaming API + Cümle Bölücü + TTS ---
    try:
        client = REGISTRY.get_genai_client(cfg.gemini_api_key)
    except Exception as e:
        err_msg = "Please enter your Google AI Studio API key in the configuration panel."
        for out_sr, audio_chunk in REGISTRY.synthesize_tts(err_msg, cfg.tts_engine, cfg.tts_voice, cfg.tts_speed):
            yield (out_sr, audio_chunk)
        return

    gemma_contents = build_gemma_contents(CHAT_HISTORY, user_text)

    t_llm_start = time.perf_counter()
    try:
        response_stream = client.models.generate_content_stream(
            model=cfg.llm_model_id,
            contents=gemma_contents,
            config=types.GenerateContentConfig(
                temperature=cfg.llm_temperature,
                max_output_tokens=cfg.llm_max_tokens,
            )
        )
    except Exception as api_err:
        err_msg = f"API error occurred with model {cfg.llm_model_id}."
        print(f"❌ AI Studio API Hatası: {api_err}")
        for out_sr, audio_chunk in REGISTRY.synthesize_tts(err_msg, cfg.tts_engine, cfg.tts_voice, cfg.tts_speed):
            yield (out_sr, audio_chunk)
        return

    buffer = ""
    full_response = ""
    approx_tokens = 0
    first_token_ms = None
    first_audio_yielded = False
    tts_first_ms = 0.0
    total_e2e_ms = 0.0
    total_tts_compute_sec = 0.0
    total_tts_audio_sec = 0.0

    for chunk in response_stream:
        delta = chunk.text if chunk.text else ""
        if not delta:
            continue

        if first_token_ms is None:
            first_token_ms = (time.perf_counter() - t_llm_start) * 1000

        # Kelime/karakter bazlı yaklaşık token hesabı (1 token ≈ 4 karakter)
        approx_tokens += max(len(delta) / 4.0, 1.0)
        buffer += delta
        full_response += delta

        # Akıllı Cümle Bölücü (Sentence Chunker)
        if re.search(r'[.!?,;]+(\s|$)', buffer) and len(buffer.split()) >= cfg.chunk_min_words:
            sentence = buffer.strip().replace("*", "")
            buffer = ""

            t_tts = time.perf_counter()
            for out_sr, audio_chunk in REGISTRY.synthesize_tts(sentence, cfg.tts_engine, cfg.tts_voice, cfg.tts_speed):
                chunk_comp_sec = time.perf_counter() - t_tts
                total_tts_compute_sec += chunk_comp_sec
                total_tts_audio_sec += len(audio_chunk) / float(out_sr)

                if not first_audio_yielded:
                    tts_first_ms = chunk_comp_sec * 1000
                    total_e2e_ms = (time.perf_counter() - pipeline_start) * 1000
                    first_audio_yielded = True
                yield (out_sr, audio_chunk)
                t_tts = time.perf_counter()

    llm_net_sec = max((time.perf_counter() - t_llm_start) - total_tts_compute_sec, 0.01)
    llm_tps = approx_tokens / llm_net_sec

    if buffer.strip():
        clean_rem = buffer.strip().replace("*", "")
        t_tts = time.perf_counter()
        for out_sr, audio_chunk in REGISTRY.synthesize_tts(clean_rem, cfg.tts_engine, cfg.tts_voice, cfg.tts_speed):
            chunk_comp_sec = time.perf_counter() - t_tts
            total_tts_compute_sec += chunk_comp_sec
            total_tts_audio_sec += len(audio_chunk) / float(out_sr)
            if not first_audio_yielded:
                tts_first_ms = chunk_comp_sec * 1000
                total_e2e_ms = (time.perf_counter() - pipeline_start) * 1000
                first_audio_yielded = True
            yield (out_sr, audio_chunk)
            t_tts = time.perf_counter()

    tts_rtf = total_tts_compute_sec / max(total_tts_audio_sec, 0.1)

    BENCHMARK.record_turn({
        "config_summary": short_profile,
        "transcript": user_text,
        "response": full_response,
        "intent": f"🧠 {intent} ({score:.2f})",
        "audio_in_sec": round(audio_in_sec, 2),
        "stt_ms": round(stt_ms, 1),
        "stt_rtf": round(stt_rtf, 4),
        "intent_ms": round(intent_ms, 1),
        "llm_ttft_ms": round(first_token_ms or 0.0, 1),
        "llm_tps": round(llm_tps, 1),
        "tts_first_ms": round(tts_first_ms, 1),
        "tts_rtf": round(tts_rtf, 4),
        "total_e2e_ms": round(total_e2e_ms, 1)
    })
    CHAT_HISTORY.append({"role": "user", "text": user_text})
    CHAT_HISTORY.append({"role": "model", "text": full_response})

# Hücre 5: AI Studio Gemma Model Seçicili WebRTC Vitrin & Benchmark Arayüzü

Bu hücrede "🔄 AI Studio'dan Gemma Modellerini Listele" butonu yer alır. API anahtarınızı girdiğinizde hesabınıza tanımlı tüm ücretsiz Gemma modellerini (gemma-3-27b-it, gemma-3-12b-it, gemma-3-4b-it, gemma-3-1b-it vb.) canlı olarak açılır menüye (Dropdown) doldurur ve her modelin gecikmesini yan paneldeki tabloda kıyaslamanızı sağlar.

In [ ]:
import gradio as gr
import numpy as np

# Başlangıçta mevcut anahtarla (varsa) Gemma listesini çek
INITIAL_GEMMA_LIST = fetch_aistudio_gemma_models(GLOBAL_CONFIG.gemini_api_key)
if GLOBAL_CONFIG.llm_model_id not in INITIAL_GEMMA_LIST and INITIAL_GEMMA_LIST:
    GLOBAL_CONFIG.llm_model_id = INITIAL_GEMMA_LIST[0]

def refresh_gemma_dropdown(api_key: str):
    GLOBAL_CONFIG.gemini_api_key = api_key.strip()
    models = fetch_aistudio_gemma_models(api_key)
    selected = models[0] if models else "gemma-3-27b-it"
    return gr.update(choices=models, value=selected), f"✅ AI Studio'dan **{len(models)} adet Gemma modeli** başarıyla listelendi!"

def apply_configuration(
    api_key, gemma_model, llm_temp, chunk_words,
    stt_model, stt_compute, stt_beam,
    embed_model, intent_thresh, fast_cache,
    tts_engine, tts_voice, tts_speed
):
    GLOBAL_CONFIG.gemini_api_key = api_key.strip()
    GLOBAL_CONFIG.llm_model_id = gemma_model
    GLOBAL_CONFIG.llm_temperature = float(llm_temp)
    GLOBAL_CONFIG.chunk_min_words = int(chunk_words)

    GLOBAL_CONFIG.stt_model_id = stt_model
    GLOBAL_CONFIG.stt_compute_type = stt_compute
    GLOBAL_CONFIG.stt_beam_size = int(stt_beam)

    GLOBAL_CONFIG.embed_model_id = embed_model
    GLOBAL_CONFIG.intent_threshold = float(intent_thresh)
    GLOBAL_CONFIG.enable_fast_path_cache = bool(fast_cache)

    GLOBAL_CONFIG.tts_engine = tts_engine
    GLOBAL_CONFIG.tts_voice = tts_voice
    GLOBAL_CONFIG.tts_speed = float(tts_speed)

    # Yerel modelleri ısıt
    REGISTRY.get_stt(stt_model, stt_compute)
    REGISTRY.get_embedder(embed_model)
    _ = list(REGISTRY.synthesize_tts("Ready.", tts_engine, tts_voice, float(tts_speed)))

    vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
    return f"✅ **Aktif Profil Güncellendi:** `{gemma_model}` + `{stt_model.split('/')[-1]}` + `{tts_engine.upper()}` (Yerel VRAM: `{vram:.2f} GB`)"

def refresh_dashboard():
    t = BENCHMARK.latest
    stats = BENCHMARK.get_percentiles()
    ram_pct = psutil.virtual_memory().percent

    hud_md = f"""
    ### ⚡ Canlı Konuşma Telemetrisi (Turn #{t['turn_id']}) — Aktif Profil: `{t['config_summary']}`
    | Katman | Gecikme (Latency) | Verimlilik / Hız (RTF & Throughput) | Hedef Eşik |
    | :--- | :--- | :--- | :--- |
    | **1. STT ({GLOBAL_CONFIG.stt_compute_type}, Beam={GLOBAL_CONFIG.stt_beam_size})** | `{t['stt_ms']} ms` | **RTF:** `{t['stt_rtf']}x` *(Ses: {t['audio_in_sec']}s)* | `RTF < 0.15x` |
    | **2. Semantic Intent Router** | `{t['intent_ms']} ms` | `{t['intent']}` | `< 25 ms` |
    | **3. AI Studio ({GLOBAL_CONFIG.llm_model_id})** | **TTFT:** `{t['llm_ttft_ms']} ms` | **Akış Hızı:** `~{t['llm_tps']} tokens/sec` | `TTFT < 350 ms` |
    | **4. TTS ({GLOBAL_CONFIG.tts_engine.upper()})** | **İlk Ses:** `{t['tts_first_ms']} ms` | **RTF:** `{t['tts_rtf']}x` | `RTF < 0.25x` |
    | **🔥 TOPLAM UÇTAN UCA (E2E)** | **`{t['total_e2e_ms']} ms`** | **P50:** `{stats['p50_e2e']} ms` | **P95:** `{stats['p95_e2e']} ms` | **`< 650 ms`** |

    **🖥️ Sistem & Maliyet:** `Yerel GPU VRAM: {t['vram_gb']} GB / 15.0 GB` • `RAM: %{ram_pct}` • `LLM Maliyeti: $0.00 (AI Studio Free Tier)`
    **🗣️ Kullanıcı:** *{t['transcript']}*
    **🤖 Gemma Yanıtı:** *{t['response']}*
    """
    return hud_md, BENCHMARK.get_summary_df()

def process_audio_fallback(audio):
    if audio is None:
        return None

    out_sr = 24000
    out_chunks = []

    for sr, chunk in configurable_voice_pipeline(audio):
        out_sr = sr
        out_chunks.append(chunk)

    if not out_chunks:
        return None

    final_audio = np.concatenate(out_chunks)
    return (out_sr, final_audio)

with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# 🎙️ Voice AI Workbench (Standart HTTP Modu)\n"
        "*WebRTC devre dışı bırakıldı. Lütfen mikrofona tıklayıp konuşun ve kaydı durdurun.*"
    )

    with gr.Row():
        # SOL PANEL
        with gr.Column(scale=5):
            with gr.Group():
                audio_in = gr.Audio(sources=["microphone"], type="numpy", label="Sesi Kaydet (Konuşmayı bitirince durdurun)")
                audio_out = gr.Audio(label="Yapay Zeka Yanıtı", autoplay=True)

            # Kayıt bitince veya dosya yüklenince otomatik çalıştır
            audio_in.stop_recording(fn=process_audio_fallback, inputs=[audio_in], outputs=[audio_out])
            audio_in.upload(fn=process_audio_fallback, inputs=[audio_in], outputs=[audio_out])

            with gr.Accordion("⚙️ Model Seçimi & Hiper-Parametre Konfigürasyonu", open=True):
                with gr.Tab("1. AI Studio (Gemma API)"):
                    api_key_tb = gr.Textbox(
                        value=GLOBAL_CONFIG.gemini_api_key,
                        type="password",
                        label="Google AI Studio API Key (GEMINI_API_KEY)",
                        placeholder="AIzaSy..."
                    )
                    with gr.Row():
                        gemma_dd = gr.Dropdown(
                            choices=INITIAL_GEMMA_LIST,
                            value=GLOBAL_CONFIG.llm_model_id,
                            label="Ücretsiz Gemma Modelleri (AI Studio)"
                        )
                        fetch_btn = gr.Button("🔄 Gemma Modellerini Listele", scale=0)
                    with gr.Row():
                        llm_temp_sl = gr.Slider(0.0, 1.0, value=0.6, step=0.1, label="Temperature")
                        chunk_sl = gr.Slider(2, 12, value=4, step=1, label="Cümle Bölücü Min. Kelime (Chunker)")

                with gr.Tab("2. STT & Intent Router"):
                    stt_model_dd = gr.Dropdown(
                        choices=[
                            "Systran/faster-distil-whisper-large-v3",
                            "Systran/faster-distil-whisper-small.en",
                            "Systran/faster-whisper-base.en"
                        ],
                        value=GLOBAL_CONFIG.stt_model_id, label="Yerel STT Modeli (CUDA)"
                    )
                    with gr.Row():
                        stt_comp_dd = gr.Dropdown(["int8_float16", "float16", "int8"], value="int8_float16", label="Quantization")
                        stt_beam_sl = gr.Slider(1, 5, value=1, step=1, label="Beam Size")

                    embed_dd = gr.Dropdown(
                        ["BAAI/bge-small-en-v1.5", "sentence-transformers/all-MiniLM-L6-v2"],
                        value=GLOBAL_CONFIG.embed_model_id, label="Intent Embedding Modeli (ONNX)"
                    )
                    with gr.Row():
                        intent_sl = gr.Slider(0.50, 0.95, value=0.80, step=0.05, label="Intent Eşik Değeri (Threshold)")
                        fast_cache_cb = gr.Checkbox(value=True, label="Fast-Path RAM Audio Cache (API Bypass)")

                with gr.Tab("3. TTS Sentezleyici"):
                    with gr.Row():
                        tts_eng_dd = gr.Dropdown(["kokoro", "piper"], value="kokoro", label="TTS Motoru (Kokoro=GPU, Piper=CPU)")
                        tts_voice_dd = gr.Dropdown(["af_heart", "af_bella", "am_adam", "am_michael"], value="af_heart", label="Kokoro Ses Profili")
                    tts_speed_sl = gr.Slider(0.8, 1.4, value=1.05, step=0.05, label="Konuşma Hızı")

                apply_btn = gr.Button("🚀 Konfigürasyonu Uygula", variant="primary")
                config_status = gr.Markdown("*Varsayılan profil aktif.*")

        # SAĞ PANEL
        with gr.Column(scale=7):
            hud_display = gr.Markdown()
            gr.Markdown("### 📈 Gemma Modelleri & Pipeline Kıyaslama Tablosu (Son 10 Konuşma)")
            bench_table = gr.Dataframe(interactive=False)

            timer = gr.Timer(1.0)
            timer.tick(fn=refresh_dashboard, outputs=[hud_display, bench_table])

    # Event bağlantıları
    fetch_btn.click(
        fn=refresh_gemma_dropdown,
        inputs=[api_key_tb],
        outputs=[gemma_dd, config_status]
    )

    apply_btn.click(
        fn=apply_configuration,
        inputs=[
            api_key_tb, gemma_dd, llm_temp_sl, chunk_sl,
            stt_model_dd, stt_comp_dd, stt_beam_sl,
            embed_dd, intent_sl, fast_cache_cb,
            tts_eng_dd, tts_voice_dd, tts_speed_sl
        ],
        outputs=config_status
    )

demo.launch(share=True, debug=True)


/tmp/ipykernel_4472/2059411158.py:84: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://258790b8b3e062e680.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
